### Load Dataset and Define Features for XGBoost Tuning

This section loads the training and testing datasets, defines the selected features, and prepares the target variable (`LogClosePrice`) for hyperparameter tuning.

The same engineered features used in the baseline model are used again here so that the tuned XGBoost model can be compared fairly with the untuned version.

In [30]:
import pandas as pd
import numpy as np
tree_features = [
    'LivingArea',
    'BathroomsTotalInteger',
    'BedroomsTotal',
    'Bed_Bath_Ratio',
    'Property_Age',
    'Months_From_Dec_2025',
    'Sqft_Per_Bedroom',
    'Lot_Utilization'
]

In [38]:
train = pd.read_csv('train_cleaned_fe.csv', engine='python', on_bad_lines='skip')
test = pd.read_csv('test_cleaned_fe.csv', engine='python', on_bad_lines='skip')

train = train.replace([np.inf, -np.inf], np.nan)
test = test.replace([np.inf, -np.inf], np.nan)

train['LogClosePrice'] = np.log(train['ClosePrice'])
test['LogClosePrice'] = np.log(test['ClosePrice'])

train = train.dropna()
test = test.dropna()

In [48]:
X_train = train[tree_features]
y_train = train['LogClosePrice']

X_test = test[tree_features]
y_test = test['LogClosePrice']

X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
xg_model = XGBRegressor()
xg_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

### Hyperparameter Tuning for XGBoost Regressor

This section applies hyperparameter tuning to the baseline XGBoost model.

`RandomizedSearchCV` is used to test different combinations of hyperparameters and find the best-performing model based on cross-validation R².

In [51]:
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV

xgb_tune_model = XGBRegressor(random_state=42)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 4, 5, 6, 8],
    'learning_rate': [0.01, 0.03, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.3, 0.5]
}

random_search = RandomizedSearchCV(
    estimator=xgb_tune_model,
    param_distributions=param_dist,
    n_iter=25,
    scoring='r2',
    cv=5,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

Fitting 5 folds for each of 25 candidates, totalling 125 fits


RandomizedSearchCV(cv=5,
                   estimator=XGBRegressor(base_score=None, booster=None,
                                          callbacks=None,
                                          colsample_bylevel=None,
                                          colsample_bynode=None,
                                          colsample_bytree=None, device=None,
                                          early_stopping_rounds=None,
                                          enable_categorical=False,
                                          eval_metric=None, feature_types=None,
                                          feature_weights=None, gamma=None,
                                          grow_policy=None,
                                          importance_type=None,
                                          interaction_constraint...
                                          n_estimators=None, n_jobs=None,
                                          num_parallel_tree=None, ...),
                   n_iter=25, n_jobs=-1,
                   param_distributions={'colsample_bytree': [0.7, 0.8, 0.9,
                                                             1.0],
                                        'gamma': [0, 0.1, 0.3, 0.5],
                                        'learning_rate': [0.01, 0.03, 0.05, 0.1,
                                                          0.2],
                                        'max_depth': [3, 4, 5, 6, 8],
                                        'min_child_weight': [1, 3, 5],
                                        'n_estimators': [100, 200, 300, 500],
                                        'subsample': [0.7, 0.8, 0.9, 1.0]},
                   random_state=42, scoring='r2', verbose=1)

### Best Hyperparameters

After running the randomized search, the best combination of hyperparameters is selected based on cross-validation performance.

These values will be used to train the final tuned XGBoost model.

In [54]:
print("Best Parameters:", random_search.best_params_)
print("Best CV Score:", random_search.best_score_)

Best Parameters: {'subsample': 0.8, 'n_estimators': 500, 'min_child_weight': 5, 'max_depth': 6, 'learning_rate': 0.01, 'gamma': 0, 'colsample_bytree': 0.7}
Best CV Score: 0.5213603360246999


In [56]:
best_xgb_model = random_search.best_estimator_
best_xgb_model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.01, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=5, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

### Evaluate the Tuned XGBoost Model

The tuned model is evaluated on both the training and testing datasets using:
- R²
- MAPE
- MdAPE

Because the dependent variable is `LogClosePrice`, the results are reported on the log scale.

In [65]:
from sklearn.metrics import r2_score, mean_absolute_percentage_error

tuned_pred_train = best_xgb_model.predict(X_train)
tuned_pred_test = best_xgb_model.predict(X_test)

tuned_r2_train = r2_score(y_train, tuned_pred_train)
tuned_r2_test = r2_score(y_test, tuned_pred_test)

tuned_mape_train = mean_absolute_percentage_error(y_train, tuned_pred_train)
tuned_mape_test = mean_absolute_percentage_error(y_test, tuned_pred_test)

tuned_mdape_train = np.median(np.abs((y_train - tuned_pred_train) / y_train))
tuned_mdape_test = np.median(np.abs((y_test - tuned_pred_test) / y_test))

print(f"Train Set -- R^2 on Log Scale: {tuned_r2_train}")
print(f"Test Set -- R^2 on Log Scale: {tuned_r2_test}")
print(f"Train Set -- MAPE: {tuned_mape_train * 100}%")
print(f"Test Set -- MAPE: {tuned_mape_test * 100}%")
print(f"Train Set -- MdAPE: {tuned_mdape_train * 100}%")
print(f"Test Set -- MdAPE: {tuned_mdape_test * 100}%")


Train Set -- R^2 on Log Scale: 0.5573917119592624
Test Set -- R^2 on Log Scale: 0.544611455802835
Train Set -- MAPE: 2.3066632208550932%
Test Set -- MAPE: 2.333771015855574%
Train Set -- MdAPE: 1.8343381817701998%
Test Set -- MdAPE: 1.8717957938542293%


In [ ]:
### Key Evaluation Metrics for the Tuned XGBoost Model

The table below summarizes the performance of the tuned XGBoost model on the training and testing datasets.

In [67]:
import pandas as pd

tuned_results_df = pd.DataFrame({
    'Metric': ['R²', 'MAPE (%)', 'MdAPE (%)'],
    'Training Set': [
        round(tuned_r2_train, 3),
        round(tuned_mape_train * 100, 3),
        round(tuned_mdape_train * 100, 3)
    ],
    'Testing Set': [
        round(tuned_r2_test, 3),
        round(tuned_mape_test * 100, 3),
        round(tuned_mdape_test * 100, 3)
    ]
})

tuned_results_df

,Metric,Training Set,Testing Set
0,R²,0.557,0.545
1,MAPE (%),2.307,2.334
2,MdAPE (%),1.834,1.872


### Interpretation

1. **Model Improvement:** Hyperparameter tuning significantly improved the performance of the XGBoost model compared to the baseline model with default settings.

2. **Higher R² Values:** The tuned model achieved an R² of approximately **0.557 on the training set** and **0.545 on the testing set**, which indicates that the model explains a much larger portion of the variance in `LogClosePrice` compared to the baseline model.

3. **Prediction Accuracy:** The MAPE and MdAPE values remain relatively low (around **2–2.3%** for MAPE and **~1.8%** for MdAPE), suggesting that the model's predictions are close to the actual property prices on average.

4. **Good Generalization:** The training and testing R² values are very close to each other, which indicates that the tuned model generalizes well to unseen data and is not significantly overfitting.

5. **Practical Meaning:** Overall, the tuned XGBoost model provides more reliable and stable predictions for residential property closing prices compared to the baseline model.